In [1]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt

In [2]:
db_path = "../data/food_price_intelligence.db"
conn = sqlite3.connect(db_path)

In [3]:
prices_sql = """
SELECT 
    dp.date,
    m.state,
    m.district,
    m.name AS market_name,
    c.name AS commodity,
    dp.min_price_inr,
    dp.max_price_inr,
    dp.modal_price_inr
FROM daily_prices dp
JOIN markets m ON dp.market_id = m.id
JOIN commodities c ON dp.commodity_id = c.id
ORDER BY dp.date ASC
"""

In [4]:
# parse_dates converts string dates into proper datetime64 objects for time-series operations
df_prices = pd.read_sql_query(prices_sql, conn, parse_dates=["date"])
df_weather = pd.read_sql_query(
    "SELECT * FROM weather_data ORDER BY date ASC", conn, parse_dates=["date"]
)
conn.close()

In [5]:
df_prices

,date,state,district,market_name,commodity,min_price_inr,max_price_inr,modal_price_inr
0,2022-01-01,Maharashtra,Nashik,Nashik_Mandi_2487,Onion,1300.0,2480.0,1850.0
1,2022-01-01,Maharashtra,Nashik,Nashik_Mandi_169,Onion,700.0,2355.0,2001.0
2,2022-01-01,Maharashtra,Nashik,Nashik_Mandi_2484,Onion,800.0,2100.0,1900.0
3,2022-01-01,Maharashtra,Nashik,Nashik_Mandi_170,Onion,300.0,2540.0,1751.0
4,2022-01-01,Maharashtra,Nashik,Nashik_Mandi_1447,Onion,400.0,2400.0,1700.0
...,...,...,...,...,...,...,...,...
52145,2026-11-13,Andhra Pradesh,Annamayya,Kalikiri APMC,Tomato,1000.0,1800.0,1400.0
52146,2026-11-13,Andhra Pradesh,Krishna,Penamaluru (HQ-Kankipadu) APMC,Paddy(Common),2441.0,2441.0,2441.0
52147,2026-11-13,Andhra Pradesh,Krishna,Penamaluru (HQ-Kankipadu) APMC,Maize,2650.0,2650.0,2650.0
52148,2026-11-13,Andhra Pradesh,Krishna,Penamaluru (HQ-Kankipadu) APMC,Black Gram(Urd Beans)(Whole),9300.0,9300.0,9300.0


In [6]:
df_prices.head()

,date,state,district,market_name,commodity,min_price_inr,max_price_inr,modal_price_inr
0,2022-01-01,Maharashtra,Nashik,Nashik_Mandi_2487,Onion,1300.0,2480.0,1850.0
1,2022-01-01,Maharashtra,Nashik,Nashik_Mandi_169,Onion,700.0,2355.0,2001.0
2,2022-01-01,Maharashtra,Nashik,Nashik_Mandi_2484,Onion,800.0,2100.0,1900.0
3,2022-01-01,Maharashtra,Nashik,Nashik_Mandi_170,Onion,300.0,2540.0,1751.0
4,2022-01-01,Maharashtra,Nashik,Nashik_Mandi_1447,Onion,400.0,2400.0,1700.0


In [7]:
df_prices["state"].unique()

<ArrowStringArray>
['Maharashtra', 'Karnataka', 'Andhra Pradesh']
Length: 3, dtype: str

In [8]:
df_columns = ["date", "state", "district", "market_name", "commodity", "min_price_inr", "max_price_inr", "modal_price_inr"]

In [9]:
df_prices["date"].dtype

dtype('<M8[us]')

In [10]:
full_range =pd.date_range(start = df_prices['date'].min(), end = df_prices['date'].max())
missing = full_range.difference(df_prices['date'])
print(f"Missing dates: {len(missing)} ")
print(df_prices["date"].count())
print(missing)

Missing dates: 392 
52150
DatetimeIndex(['2024-08-01', '2024-08-02', '2024-08-03', '2024-12-14',
               '2024-12-15', '2024-12-16', '2024-12-17', '2024-12-18',
               '2024-12-19', '2024-12-20',
               ...
               '2026-11-03', '2026-11-04', '2026-11-05', '2026-11-06',
               '2026-11-07', '2026-11-08', '2026-11-09', '2026-11-10',
               '2026-11-11', '2026-11-12'],
              dtype='datetime64[us]', length=392, freq=None)


In [11]:
df_weather.head()

,id,region_name,date,temperature_max_c,temperature_min_c,precipitation_mm
0,2047,Nashik,2022-01-01,26.4,14.0,0.0
1,3145,Kolar,2022-01-01,25.1,17.8,3.9
2,4243,Andhra Pradesh,2022-01-01,30.8,20.7,0.0
3,2048,Nashik,2022-01-02,25.9,14.8,0.2
4,3146,Kolar,2022-01-02,25.3,16.2,0.0


In [14]:
full_range =pd.date_range(start = df_weather['date'].min(), end = df_weather['date'].max())
missing = full_range.difference(df_weather['date'])
print(f"Missing dates: {len(missing)} ")
print(df_weather["date"].count())
print(missing)

Missing dates: 0 
5340
DatetimeIndex([], dtype='datetime64[us]', freq='D')


In [12]:
audit_summary = (
    df_prices.groupby(["state", "commodity"])
    .agg(
        total_rows=("modal_price_inr", "count"),
        unique_markets=("market_name", "nunique"),
        earliest_date=("date", "min"),
        latest_date=("date", "max"),
        avg_modal_price=("modal_price_inr", "mean"),
    )
    .reset_index()
    .sort_values(by=["state", "total_rows"], ascending=[True, False])
)

In [13]:
df_prices.head()

,date,state,district,market_name,commodity,min_price_inr,max_price_inr,modal_price_inr
0,2022-01-01,Maharashtra,Nashik,Nashik_Mandi_2487,Onion,1300.0,2480.0,1850.0
1,2022-01-01,Maharashtra,Nashik,Nashik_Mandi_169,Onion,700.0,2355.0,2001.0
2,2022-01-01,Maharashtra,Nashik,Nashik_Mandi_2484,Onion,800.0,2100.0,1900.0
3,2022-01-01,Maharashtra,Nashik,Nashik_Mandi_170,Onion,300.0,2540.0,1751.0
4,2022-01-01,Maharashtra,Nashik,Nashik_Mandi_1447,Onion,400.0,2400.0,1700.0


In [15]:
df_weather.tail()

,id,region_name,date,temperature_max_c,temperature_min_c,precipitation_mm
5335,5,Kolar,2026-11-13,26.8,15.0,0.0
5336,6,Andhra Pradesh,2026-11-13,30.8,22.4,0.7
5337,7,Nashik,2026-11-13,30.1,17.8,0.0
5338,8,Kolar,2026-11-13,26.8,15.0,0.0
5339,9,Andhra Pradesh,2026-11-13,30.8,22.4,0.7
